# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Classification

My lane is a classification problem because the goal is to predict whether a content page is declining or not declining.

Each page belongs to one of two categories:

- Declining (1)
- Not Declining (0)

The model uses features such as content age, impressions, CTR, average position, and days since the last update to classify pages. This prediction helps identify which pages should be prioritized for content refresh and optimization.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or Proxy

I would predict whether a content page is declining or not declining.

Target: is_declining_label

Values:
- 1 = Declining page
- 0 = Not Declining page

The label comes from an observed outcome. In the dataset, pages are labeled as declining when their trend_direction is "down". This creates the target variable is_declining_label, which the model learns to predict using content and performance features.

The goal is to identify pages that are likely declining so they can be prioritized for content refresh and optimization.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down")
).astype(int)

df[["trend_direction", "is_declining_label"]].head()

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*
### Success Metric

The success metric is Precision@50.

Precision@50 measures the percentage of truly declining pages among the top 50 pages flagged by the model.

A good model should have a higher Precision@50 than the hand-written rule baseline because it means more of the pages selected for review are actually declining.

For example, a Precision@50 of 0.70 means that 35 out of the top 50 pages identified by the model are truly declining pages.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check class distribution

declining_rate = df["is_declining_label"].mean()

print(f"Declining pages: {declining_rate:.3f}")
print(f"Non-declining pages: {1 - declining_rate:.3f}")


Declining pages: 0.542
Non-declining pages: 0.458


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
### Unit of Analysis

One row represents one content page.

Each row contains information about a single page, including metrics such as content age, impressions, CTR, average position, word count, and trend direction.

The model makes one prediction per page: whether that page is declining or not declining.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("One row = one content page")
print("Dataset shape:", df.shape)

df.head()


One row = one content page
Dataset shape: (30000, 45)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
### Why ML Beats a Fixed Rule Here

A fixed rule such as "refresh pages older than 180 days" is too simple because page performance depends on multiple factors at once. Some old pages still perform well, while some newer pages may be declining.

The relationship between impressions, CTR, average position, content age, and update frequency is not captured by a single threshold. Machine learning can combine these signals and identify patterns that are difficult to express with an if-statement.

This makes ML a better decision-support tool for prioritizing content refresh opportunities.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Examine how features relate to declining content

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

df[features + ["is_declining_label"]].corr()["is_declining_label"].sort_values(ascending=False)

,is_declining_label
is_declining_label,1.000000
word_count,0.090157
days_since_last_update,0.081383
impressions_90d,-0.018175
avg_position,-0.029035
ctr,-0.061911
content_age_days,-0.163882


In [23]:
corrs = df[features + ["is_declining_label"]].corr()["is_declining_label"]
print(corrs.sort_values(ascending=False))

is_declining_label        1.000000
word_count                0.090157
days_since_last_update    0.081383
impressions_90d          -0.018175
avg_position             -0.029035
ctr                      -0.061911
content_age_days         -0.163882
Name: is_declining_label, dtype: float64


The correlations show that no single feature perfectly explains whether a page is declining. Several features have small relationships with the target, suggesting that the model needs to consider multiple signals together. This supports the use of machine learning instead of a simple fixed rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.